---
title: Pemetaan perkembangan bukaan tambang laterit di Pulau Gag, Raja Ampat
short_title: Bukaan tambang Pulau Gag
subject: Studi Kasus
subtitle: Melacak luas dan arah perluasan lahan terbuka laterit (2017–2025) dari komposit tahunan Sentinel-2 GeoMAD.
description: Menggunakan komposit tahunan bebas awan Sentinel-2 GeoMAD (2017–2025) bersama penentuan ambang batas otomatis Otsu untuk memetakan sebaran, laju pembukaan baru, dan arah perluasan tambang laterit di Pulau Gag, Raja Ampat.
keywords:
  - sentinel-2
  - geomad
  - pertambangan
  - laterit
  - pulau-gag
  - raja-ampat
  - otsu
  - perubahan-lahan
---

## A. Latar belakang

Pembukaan lahan pertambangan nikel laterit secara terbuka (*open-pit*) di pulau kecil tropis berpotensi menyusutkan tutupan hutan sekaligus memicu erosi dan limpasan sedimen ke perairan dangkal di sekitarnya. Pada tambang nikel di wilayah tropis, pengupasan hutan menyingkap lapisan tanah pelapukan berwarna cokelat-kemerahan yang kaya mineral oksida besi, dikenal sebagai **tanah laterit**. Ciri warna kemerahan inilah yang membedakan tanah galian tambang dari pasir pantai putih maupun permukaan atap permukiman pada citra satelit.

Pulau Gag di Kabupaten Raja Ampat, Papua Barat Daya, menjadi salah satu lokasi yang disorot dalam pengawasan pertambangan pulau kecil pada 2025. Untuk mendukung pengawasan tersebut dengan bukti spasial yang terukur, analisis ini memanfaatkan komposit tahunan bebas awan *Geometric Median* Sentinel-2 (`s2_geomad_annual`, sejak awal arsip 2017 hingga 2025) [(Roberts dkk., 2017)](#ref-roberts-2017) untuk menjawab berapa luas bukaan tanah laterit dari tahun ke tahun, kapan laju pembukaan barunya paling cepat, ke arah mana perluasannya bergerak, dan apakah pembukaan masih berlanjut hingga 2025.

## B. Menyiapkan data dan wilayah kajian

Seluruh pustaka yang dibutuhkan disiapkan terlebih dahulu, termasuk `Datacube` untuk memuat komposit `s2_geomad_annual` dan `threshold_otsu` untuk penentuan ambang batas spektral secara otomatis.

In [ ]:
import folium
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import odc.geo.xr
import pandas as pd
import requests
import xarray as xr
from datacube import Datacube
from ipyleaflet import basemaps
from matplotlib.colors import BoundaryNorm, ListedColormap
from odc.geo.geom import Geometry
from odc.geo.xr import rasterize
from skimage.filters import threshold_otsu

Batas daratan Pulau Gag diambil dari OpenStreetMap (`way 26945718`) melalui API Nominatim agar perhitungan luas dan pemotongan piksel tepat mengikuti garis pantai pulau [(OpenStreetMap contributors, t.t.)](#ref-openstreetmap).

In [ ]:
response = requests.get(
    "https://nominatim.openstreetmap.org/lookup",
    params={"osm_ids": "W26945718", "format": "geojson", "polygon_geojson": 1},
    headers={"User-Agent": "piksel-studi-kasus-tambang/1.0"},
    timeout=20,
)
response.raise_for_status()

aoi = gpd.GeoDataFrame.from_features(response.json()["features"], crs="EPSG:4326")
aoi_utm = aoi.to_crs("EPSG:32752")
print(f"Luas poligon Pulau Gag: {aoi_utm.area.sum() / 10_000:.1f} ha")

aoi.explore(tiles=basemaps.Esri.WorldImagery)

*Sumber batas:* © OpenStreetMap contributors, ODbL [(OpenStreetMap contributors, t.t.)](#ref-openstreetmap).

Selanjutnya, komposit tahunan `s2_geomad_annual` (2017–2025) dimuat pada proyeksi `EPSG:32752` dengan resolusi 10 meter.

In [ ]:
PIXEL_AREA_HA = 0.01  # 10 m x 10 m = 100 m^2 = 0,01 ha
RGB_BRIGHTNESS_MAX = 2500.0  # Rentang tampilan 0-0,25 pada reflektansi skala 0-10.000

dc = Datacube(app="studi_kasus_tambang_gag")
odc_geom = Geometry(aoi.geometry.iloc[0], crs="EPSG:4326")

ds = dc.load(
    product="s2_geomad_annual",
    measurements=["red", "green", "blue", "nir", "swir16"],
    geopolygon=odc_geom,
    time=("2017", "2025"),
    output_crs="EPSG:32752",
    resolution=(-10, 10),
    dask_chunks={"time": 1, "x": 1024, "y": 1024},
).compute()

ds

Nilai reflektansi permukaan pada dataset ini disimpan dalam tipe bilangan bulat `uint16` dengan faktor skala `0–10.000` (bernilai `0` pada piksel tanpa data), sehingga perlu dipotong ke batas pulau lalu dikonversi ke `float32` sebelum menghitung rasio spektral maupun menampilkan citra RGB.

In [ ]:
island_mask = rasterize(odc_geom.to_crs("EPSG:32752"), ds.odc.geobox) > 0
valid = (ds["red"] > 0) & island_mask

red = ds["red"].where(valid).astype("float32")
green = ds["green"].where(valid).astype("float32")
blue = ds["blue"].where(valid).astype("float32")
nir = ds["nir"].where(valid).astype("float32")
swir1 = ds["swir16"].where(valid).astype("float32")

Lima band reflektansi untuk periode 2017–2025 kini sudah siap di dalam batas daratan Pulau Gag. Namun, angka pantulan cahaya mentah pada masing-masing band belum secara langsung memisahkan mana area hutan, pantai, maupun galian tambang. Agar analisis tren ekspansi dan risiko erosi pada tahap berikutnya bertumpu pada peta yang bersih, langkah selanjutnya mengolah kelima band tersebut menjadi penanda tahunan lahan terbuka laterit.

## C. Identifikasi tanah laterit terbuka

Untuk memetakan bukaan tambang secara konsisten antartahun, kita perlu mendefinisikan ciri spektral tanah laterit yang memisahkannya baik dari tajuk hutan maupun dari permukaan terbuka lain seperti pasir pantai berkarang. Tiga indeks spektral standar dihitung dari reflektansi permukaan:

$$
\text{NDVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red}}
$$

$$
\text{BSI} = \frac{(\text{SWIR1} + \text{Red}) - (\text{NIR} + \text{Blue})}{(\text{SWIR1} + \text{Red}) + (\text{NIR} + \text{Blue})}
$$

$$
\text{Rasio Oksida Besi} = \frac{\text{Red}}{\text{Blue}}
$$

Rumus di atas merujuk pada literatur untuk *NDVI* [(Rouse dkk., 1974)](#ref-rouse-1974), *BSI* [(Rikimaru dkk., 2002)](#ref-rikimaru-2002); [(Diek dkk., 2017)](#ref-diek-2017), dan rasio oksida besi [(Segal, 1982)](#ref-segal-1982). Ketiga indeks bekerja saling melengkapi: `NDVI` memisahkan vegetasi dari lahan tidak bervegetasi, `BSI` menonjolkan tanah terbuka, sedangkan rasio `Red / Blue` menonjolkan tanah laterit kaya oksida besi (memantulkan band merah kuat dan menyerap biru) sehingga pasir pantai maupun atap bangunan tidak ikut terbaca sebagai tambang.

Agar batas pemisah ketiga indeks tidak ditentukan lewat tebakan angka manual, ambang batas dihitung langsung dari distribusi piksel Pulau Gag tahun 2025 menggunakan metode **Otsu** [(Otsu, 1979)](#ref-otsu-1979):

1. Pada rasio `Red / Blue`, ambang Otsu global dihitung langsung di seluruh pulau untuk memisahkan tanah merah laterit dari permukaan lain.
2. Pada `NDVI` dan `BSI`, karena sebagian besar daratan Pulau Gag masih berupa hutan rapat, metode Otsu diterapkan dua tahap: tahap pertama memisahkan area bukan hutan rapat berdasarkan `NDVI`, lalu di dalam area bukan hutan tersebut Otsu dijalankan kembali pada `NDVI` dan `BSI` guna memisahkan tanah gundul dari semak atau vegetasi jarang.

Langkah pertama menghitung ketiga indeks spektral untuk seluruh seri tahun 2017–2025.

In [ ]:
bsi = ((swir1 + red) - (nir + blue)) / ((swir1 + red) + (nir + blue))
ndvi = (nir - red) / (nir + red)
iron_ratio = red / blue

Selanjutnya, nilai ambang batas diturunkan otomatis dari sebaran piksel tahun 2025 menggunakan metode Otsu [(Otsu, 1979)](#ref-otsu-1979), lalu langsung digabungkan menjadi lapisan tahunan `laterite_open`.

In [ ]:
b25 = bsi.isel(time=-1).values
n25 = ndvi.isel(time=-1).values
i25 = iron_ratio.isel(time=-1).values
m25 = np.isfinite(b25) & np.isfinite(n25) & np.isfinite(i25)

t_iron = float(threshold_otsu(np.clip(i25[m25], 0.0, 5.0)))
t_forest = float(threshold_otsu(n25[m25]))
non_forest = m25 & (n25 < t_forest)
t_ndvi = float(threshold_otsu(n25[non_forest]))
t_bsi = float(threshold_otsu(b25[non_forest]))

laterite_open = (bsi > t_bsi) & (ndvi < t_ndvi) & (iron_ratio > t_iron)

print(f"Batas hutan NDVI < {t_forest:.3f}")
print(f"Tanah terbuka: NDVI < {t_ndvi:.3f}, BSI > {t_bsi:.3f}, Red/Blue > {t_iron:.3f}")

Hasil pendeteksian otomatis tersebut kemudian digambarkan di atas citra *true color* 2025 untuk memeriksa secara visual apakah batasnya benar-benar mengikuti galian tanah cokelat-kemerahan dan jalan tambang.

In [ ]:
ds_2025 = ds.isel(time=-1)
mask_2025 = laterite_open.isel(time=-1).values
area_2025 = float(mask_2025.sum()) * PIXEL_AREA_HA

rgb_2025 = np.stack(
    [ds_2025["red"].values, ds_2025["green"].values, ds_2025["blue"].values],
    axis=-1,
).astype("float32")
rgb_2025 = np.clip(rgb_2025 / RGB_BRIGHTNESS_MAX, 0.0, 1.0)
rgb_2025[~island_mask.values] = 1.0

fig, ax = plt.subplots(figsize=(7, 8), constrained_layout=True)
ax.imshow(rgb_2025)
ax.contour(mask_2025.astype(float), levels=[0.5], colors="#fde047", linewidths=1.0)

ax.set_title("Verifikasi Threshold", fontsize=11)
ax.axis("off")
plt.show()

Garis batas kuning selaras dengan galian tanah merah-cokelat dan jalur angkut tambang, sehingga ambang batas ini dapat digunakan untuk tahap analisis berikutnya.

## D. Perkembangan bukaan tambang (2017–2025)

Untuk melihat laju perluasan hingga 2025, setiap tahun perlu dihitung luas lahan **terbuka aktif** dan **bukaan baru**. Terbuka aktif merujuk pada luasan lahan terbuka pada tahun tersebut, sedangkan bukaan baru adalah area yang baru pertama kali terdeteksi terbuka dan belum pernah terbuka pada tahun-tahun sebelumnya.

In [ ]:
years = [int(str(t)[:4]) for t in ds.time.values]
baseline_open_2017 = laterite_open.isel(time=0).values

first_open_year = np.zeros(baseline_open_2017.shape, dtype=np.int16)
first_open_year[baseline_open_2017] = 2017

cum_ever = baseline_open_2017.copy()
records = [
    {
        "Tahun": 2017,
        "Terbuka aktif (ha)": round(float(baseline_open_2017.sum() * PIXEL_AREA_HA), 1),
        "Bukaan baru (ha)": 0.0,
        "Akumulasi area terbuka (ha)": round(float(cum_ever.sum() * PIXEL_AREA_HA), 1),
    }
]

for i, yr in enumerate(years[1:], start=1):
    open_yr = laterite_open.isel(time=i).values
    newly_opened = open_yr & (~cum_ever)
    first_open_year[newly_opened] = yr
    cum_ever |= newly_opened
    records.append(
        {
            "Tahun": yr,
            "Terbuka aktif (ha)": round(float(open_yr.sum() * PIXEL_AREA_HA), 1),
            "Bukaan baru (ha)": round(float(newly_opened.sum() * PIXEL_AREA_HA), 1),
            "Akumulasi area terbuka (ha)": round(float(cum_ever.sum() * PIXEL_AREA_HA), 1),
        }
    )

summary_df = pd.DataFrame(records)
summary_df

Untuk melihat pola percepatan maupun perlambatannya lebih jelas, tren luasan tahunan dari tabel di atas divisualisasikan terlebih dahulu ke dalam grafik batang dan garis.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8), constrained_layout=True)

ax.bar(summary_df["Tahun"], summary_df["Terbuka aktif (ha)"], color="#cbd5e1", label="Terbuka aktif")
ax.bar(summary_df["Tahun"], summary_df["Bukaan baru (ha)"], color="#dc2626", width=0.5, label="Bukaan baru per tahun")
ax.plot(
    summary_df["Tahun"],
    summary_df["Akumulasi area terbuka (ha)"],
    color="#7f1d1d",
    marker="o",
    linewidth=2,
    label="Akumulasi area terbuka",
)
ax.set_xticks(years)
ax.set_ylabel("Luas (ha)")
ax.set_title("Tren luas terbuka aktif dan bukaan baru (2017–2025)")
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.legend(frameon=False, loc="upper left")
plt.show()

Bukaan baru memuncak pada **2023** (**52,2 ha**), sempat melandai pada **2024** (**10,1 ha**) lalu kembali naik pada **2025** (**25,9 ha**), sementara luas terbuka aktif bertahan di kisaran **231 ha** dan akumulasi area terbuka terus bertambah hingga **314,9 ha**.

Selanjutnya, penambahan bukaan baru diplotkan ke peta untuk melihat di area mana saja perluasan terjadi.

In [ ]:
year_cmap = plt.get_cmap("turbo", len(years))
year_norm = BoundaryNorm(np.arange(2016.5, 2026.5, 1.0), len(years))

fig, ax = plt.subplots(figsize=(7, 8), constrained_layout=True)
bg = np.where(
    island_mask.values,
    np.clip(ds["red"].isel(time=-1).values / RGB_BRIGHTNESS_MAX, 0.15, 0.85),
    1.0,
)
ax.imshow(bg, cmap="gray", vmin=0, vmax=1)

masked_year = np.ma.masked_where(first_open_year == 0, first_open_year)
im = ax.imshow(masked_year, cmap=year_cmap, norm=year_norm)

cbar = fig.colorbar(im, ax=ax, ticks=years, fraction=0.045, pad=0.03)
ax.set_title("Lokasi penambahan bukaan tambang (2017–2025)")
ax.axis("off")
plt.show()

Aktivitas pembukaan lahan mulai tampak pada 2018–2019, diikuti indikasi pembukaan jalur menuju bagian tengah pulau pada 2020. Setelahnya, bukaan lahan makin lama makin masuk ke dalam pulau dari arah timur menuju ke barat.

## E. Peta interaktif sebaran bukaan tahunan

In [ ]:
def to_mask_da(arr):
    da = xr.DataArray(arr.astype("float32"), coords={"y": ds.y, "x": ds.x}, dims=("y", "x"))
    da = da.odc.assign_crs(ds.odc.crs)
    return da.where(da == 1)


open_2017_2018 = (first_open_year >= 2017) & (first_open_year <= 2018)
open_2019_2023 = (first_open_year >= 2019) & (first_open_year <= 2023)
open_2024_2025 = first_open_year >= 2024

m = aoi.explore(
    color="#f8fafc",
    style_kwds={"fillOpacity": 0.0, "weight": 2},
    name="Batas Pulau Gag",
    tiles=basemaps.Esri.WorldImagery,
)

to_mask_da(open_2017_2018).odc.explore(
    map=m,
    name="Bukaan awal (2017-2018)",
    cmap=ListedColormap(["#2563eb"]),
    vmin=0,
    vmax=1,
    opacity=0.8,
)
to_mask_da(open_2019_2023).odc.explore(
    map=m,
    name="Perluasan utama (2019–2023)",
    cmap=ListedColormap(["#f97316"]),
    vmin=0,
    vmax=1,
    opacity=0.8,
)
to_mask_da(open_2024_2025).odc.explore(
    map=m,
    name="Bukaan terbaru (2024–2025)",
    cmap=ListedColormap(["#dc2626"]),
    vmin=0,
    vmax=1,
    opacity=0.85,
)
folium.LayerControl(collapsed=False).add_to(m)
m

Melalui panel kontrol lapisan di pojok kanan atas, setiap periode bukaan dapat dinyalakan secara bergantian untuk membandingkan posisi awal tambang dengan area perluasan terbarunya. Perlu diingat bahwa waktu perekaman citra latar *Esri World Imagery* dapat berbeda dari komposit GeoMAD 2025.

## F. Pengembangan analisis untuk pemantauan pertambangan

Alur pemetaan bukaan tambang berbasis `s2_geomad_annual` ini dapat diterapkan di wilayah pertambangan lain atau dikembangkan ke beberapa arah berikut:

- **Menyandingkan dengan batas izin resmi dan kawasan hutan.** Menggabungkan peta bukaan tahunan bersama batas Kontrak Karya atau IUP, rencana kerja (RKAB), serta peta status kawasan hutan (misalnya Persetujuan Penggunaan Kawasan Hutan/PPKH) membantu memeriksa apakah perluasan bukaan tetap berada di dalam koridor kerja yang diizinkan.
- **Menarik garis dasar lebih awal dengan arsip Landsat.** Karena `s2_geomad_annual` tersedia mulai 2017, riwayat pembukaan sebelum periode tersebut dapat ditelusuri menggunakan seri Landsat (tersedia sejak 2013 untuk `ls8_c2l2_sr` maupun arsip Landsat sebelumnya) agar kondisi rona awal pulau sebelum ekspansi terlihat lebih utuh.
- **Memanfaatkan citra resolusi sub-meter atau foto udara drone.** Citra 10 meter Sentinel-2 memadai untuk memetakan jejak bukaan secara menyeluruh, tetapi pemisahan detail fasilitas seperti batas teras pit, kolam pengendap (*sediment pond*), serta kondisi tanaman reklamasi memerlukan ortofoto drone atau citra satelit resolusi sangat tinggi.
- **Memodelkan jalur limpasan sedimen dengan DEM lokal mutakhir.** Jika pengawasan ingin diperluas ke risiko aliran lumpur menuju pesisir, analisis perlu memakai model permukaan tanah (*DTM*) resolusi tinggi dari LiDAR atau fotogrametri drone yang merekam kontur teras tambang terkini, bukan DEM global lama yang masih memuat tinggi kanopi.

## Referensi

(ref-roberts-2017)=
1. Roberts, D., Mueller, N., & McIntyre, A. (2017). *High-dimensional pixel composites from Earth observation time series*. *IEEE Transactions on Geoscience and Remote Sensing*, 55(11), 6254–6264. https://doi.org/10.1109/TGRS.2017.2723896

(ref-rouse-1974)=
2. Rouse, J. W., Jr., Haas, R. H., Schell, J. A., & Deering, D. W. (1974). *Monitoring vegetation systems in the Great Plains with ERTS*. *Third ERTS Symposium*, NASA SP-351, 309–317.

(ref-rikimaru-2002)=
3. Rikimaru, A., Roy, P. S., & Miyatake, S. (2002). *Tropical forest cover density mapping*. *Tropical Ecology*, 43(1), 39–47.

(ref-diek-2017)=
4. Diek, S., Fornallaz, F., Schaepman, M., & De Jong, R. (2017). *Barest pixel composite for agricultural areas using Landsat time series*. *Remote Sensing*, 9(12), 1245. https://doi.org/10.3390/rs9121245

(ref-segal-1982)=
5. Segal, D. B. (1982). *Theoretical basis for differentiation of ferric-iron bearing minerals, using Landsat MSS data*. *Proceedings of the International Symposium on Remote Sensing of Environment, 2nd Thematic Conference on Remote Sensing for Exploratory Geology*, Fort Worth, TX, 949–951.

(ref-otsu-1979)=
6. Otsu, N. (1979). *A Threshold Selection Method from Gray-Level Histograms*. *IEEE Transactions on Systems, Man, and Cybernetics*, 9(1), 62–66. https://doi.org/10.1109/TSMC.1979.4310076

(ref-openstreetmap)=
7. OpenStreetMap contributors. (n.d.). *Pulau Gag* (way 26945718). Data tersedia di bawah lisensi ODbL.